# Construire un projet de Machine Learning:

## Partie 1 | `Préparation des données (Data Wrangling)`

Dans ce cours, nous allons construire un projet de machine learning de bout en bout en Python. 

### Contenu du cours
Nous couvrirons cela dans cinq notebooks/applications distincts :
1. **Opérations sur les données** - Ingestion des données, préparation des données et écriture dans Snowflake avec Modin (`modin.pandas`) et Snowpark (`snowflake-snowpark-python`)
2. **Analyse exploratoire des données** - Exploration des données, statistiques descriptives et visualisation avec `Altair` et `Streamlit`
3. **Machine Learning (ML)** - Préparation des données et des variables pour construire des modèles avec différents algorithmes ML (Régression Logistique, Random Forest et Support Vector Machine) avec `scikit-learn`
4. **Suivi des expériences** - Lancement du suivi des expériences lors de la construction et du test de différents hyperparamètres avec `ExperimentTracking()` de `snowflake-ml-python`
5. **Application de données** - Construction d’une application de données partageable avec `Streamlit`

### Ce que nous allons couvrir (dans ce notebook) :

1. **Chargement et préparation des données** - Charger le dataset des ours et le préparer pour l’analyse avec Modin (`modin.pandas`) et Snowpark (`snowflake-snowpark-python`)
2. **Statistiques de base** - Calculer et visualiser les statistiques descriptives du dataset
3. **Analyse de distribution des variables** - Explorer la distribution des différentes caractéristiques selon les espèces d’ours avec `Altair` et `Streamlit`
4. **Analyse de corrélation** - Étudier les relations entre les variables numériques avec des cartes de chaleur de corrélation avec `Altair` et `Streamlit`
5. **Relations entre variables** - Visualiser les relations entre paires de variables à l’aide de graphiques de dispersion interactifs avec `Altair` et `Streamlit`
6. **Analyse des variables catégorielles** - Examiner la distribution des variables catégorielles incluant la classification des espèces avec `Altair` et `Streamlit`



# Configuration du notebook

## Installer les bibliothèques prérequis

Les Snowflake Notebooks incluent par défaut plusieurs bibliothèques Python courantes. Pour en ajouter d'autres, utilisez le menu **Packages** en haut à droite. 

Ajoutons les packages suivants :
- `modin` - Effectuer des opérations de données (lecture/écriture) et la préparation des données comme avec pandas via l’API [Snowpark pandas](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/modin/index)
- `scikit-learn` - Réaliser des divisions de données et construire des modèles de machine learning

# Configuration des données

Dans cette étape, nous allons effectuer les actions suivantes :
1. Charger des données tabulaires depuis un dépôt GitHub
2. Créer un stage Snowflake pour stocker les données d’image

## Dataset des ours

Le dataset correspond à un problème classique de classification multi-classe où l’objectif du machine learning est de classer chaque entrée dans l’une des quatre espèces d’ours à partir de ses caractéristiques.

Le dataset des ours comprend 200 échantillons et chaque entrée est décrite par 6 caractéristiques différentes relatives aux caractéristiques physiques de l’ours (également appelées paramètres, variables indépendantes ou variables X) et est associée à l’une des quatre espèces (A, B, C et D).



1. **Données tabulaires (`bear_raw_data.csv`)** : La première partie des données contient les mesures physiques de chaque ours, représentées par les colonnes suivantes :

| Colonne                    | Description                                                              |
| -------------------------- | ------------------------------------------------------------------------ |
| `ID`                       | Un identifiant unique pour chaque ours.                                  |
| `Species`                  | L’espèce de l’ours.                                                      |
| `Body_Mass_kg`             | La masse corporelle de l’ours, mesurée en kilogrammes (kg).              |
| `Shoulder_Hump_Height_cm`  | La hauteur de la bosse d’épaule de l’ours, mesurée en centimètres (cm).  |
| `Claw_Length_cm`           | La longueur des griffes de l’ours, mesurée en centimètres (cm).          |
| `Snout_Length_cm`          | La longueur du museau de l’ours, mesurée en centimètres (cm).            |
| `Forearm_Circumference_cm` | La circonférence de l’avant-bras de l’ours, mesurée en centimètres (cm). |
| `Ear_Length_cm`            | La longueur des oreilles de l’ours, mesurée en centimètres (cm).         |


## Charger les données

Ici, nous allons charger la première partie du dataset qui comprend l’identifiant, l’espèce de l’ours et 6 colonnes de caractéristiques :
- id
- species
- body_mass_kg
- shoulder_hump_height_cm
- claw_length_cm
- snout_length_cm
- forearm_circumference_cm
- ear_length_cm

Concernant la seconde partie, nous la préparerons à partir de caractéristiques extraites d’une collection d’images d’ours correspondant à chaque identifiant de ligne.

In [ ]:
import modin.pandas as pd
from modin.config import AutoSwitchBackend; AutoSwitchBackend.disable()
import snowflake.snowpark.modin.plugin

df = pd.read_csv("s3://logbrain-datalake/datasets/bears-data/bear_raw_data.csv")
df

## Charger les images

Comme mentionné précédemment, chaque ligne du dataset possède un identifiant unique pour chaque ours ainsi qu’une image correspondante nommée avec cet identifiant (par ex. `GRZ_01`, `GRZ_02`, `GRZ_03`, etc.)

### Créer un stage Snowflake pour stocker les images téléchargées

Avant de pouvoir travailler avec une image, nous devons créer un stage Snowflake pour stocker les images.

Nous pouvons le faire via l’interface Snowsight ou avec une instruction SQL.

Concrètement, cela se fait en 3 étapes :
1. Créer le stage avec `CREATE STAGE stage_name`
2. Activer `DIRECTORY` afin que les fichiers soient visibles dans le stage
3. Utiliser le chiffrement côté serveur afin de pouvoir exploiter les fonctions Cortex sur les images stockées dans le stage

Avant de créer le stage, créons notre base de données de travail (ici j’utiliserai `bears_data` ; vous pouvez bien sûr la remplacer par une autre base de votre choix).

Je préciserai également que nous utiliserons le schéma `stages`.

In [ ]:
CREATE DATABASE IF NOT EXISTS bears_data;
CREATE SCHEMA IF NOT EXISTS stages;

USE DATABASE bears_data;
USE SCHEMA stages;

In [ ]:



CREATE OR REPLACE STAGE bears_stage
  DIRECTORY = ( ENABLE = true )
  ENCRYPTION = ( TYPE = 'SNOWFLAKE_SSE' );

Ensuite, rendez-vous dans l’explorateur de base de données de Snowsight et téléversez les 200 images d’ours dans le stage `BEAR_STAGE` situé dans la base `BEARS_DATA` et le schéma `STAGE`.

Revenez ensuite dans ce notebook et exécutez la commande `ls` pour interroger le stage `@bear`.

In [ ]:
ls @bears_stage

## Afficher les images des ours

Maintenant que toutes les images ont été téléversées, examinons-les.

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

st.title("Bear species")

# Create a single row with 4 columns
cols = st.columns(4)

# Bear species and their captions
bears = [
    ("ABB", "American Black Bear"),
    ("EUR", "Eurasian Brown Bear"), 
    ("GRZ", "Grizzly Bear"),
    ("KDK", "Kodiak Bear")
]

# Display images in grid using loop
for col, (species, caption) in zip(cols, bears):
    with col:
        st.image(
            f'https://logbrain-datalake.s3.eu-west-1.amazonaws.com/datasets/bears-data/images/{species}_01.png?raw=true',
            caption=f"{caption} ({species}_01.png)"
        )


# Analyse d’images

Pourquoi analysons-nous les images ? Comme mentionné plus haut, nous allons ajouter des caractéristiques supplémentaires au dataset en analysant les images d’ours afin de déterminer :
- La couleur du pelage
- Le profil facial
- La texture des coussinets

Ces 3 caractéristiques sont ajoutées au dataset chargé ci-dessus dans la cellule `py_load_data` et stockées dans la variable `df`.

## Inférence LLM sur l’image

Pour effectuer une inférence avec un LLM, nous réalisons les 4 choses suivantes :
1. Utiliser la fonction SQL `AI_COMPLETE()` pour analyser l’image
2. Utiliser le modèle LLM `claude-3-5-sonnet` pour réaliser l’inférence
3. Spécifier le prompt qui fournira les instructions nécessaires pour analyser l’image
4. Utiliser `TO_FILE()` pour spécifier le fichier image à analyser, tout en fournissant le stage et le nom du fichier comme paramètres d’entrée.

In [ ]:
SELECT AI_COMPLETE(
  'claude-3-5-sonnet',
  'What is the fur color of the bear?',
  TO_FILE('@bears_stage', 'ABB_01.png')
);

## Passer de requêtes statiques à des requêtes dynamiques

Nous allons maintenant faire essentiellement la même chose que ci-dessus, mais en structurant cela de manière à pouvoir passer des variables Python dans la requête SQL, ce qui la rend plus dynamique et réutilisable dans un workflow programmatique.

En pratique, cela nous permettra de traiter de façon itérative un ensemble de 200 images.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

prompt = 'What is the fur color of the bear?'
image = 'ABB_01.png'

query = f"""
SELECT AI_COMPLETE('claude-3-5-sonnet',
    '{prompt}',
    TO_FILE('@bears_stage', '{image}'));
"""

session.sql(query)

En développant la requête ci-dessus, nous allons maintenant modifier le prompt afin d’inférer la couleur du pelage de l’ours à partir de son image.

In [ ]:
prompt = """
Analyze the provided image of a bear. Describe only the fur color of the bear 
by choosing the most appropriate term from the following list. The response 
should be a single value.
- Light Brown
- Medium Brown
- Blond
- Dark Brown
- Grizzled (A mix of colors with silver-tipped hairs)
- Reddish Brown
- Blackish Brown
- Black
- Brown
- Cinnamon
"""

image = 'ABB_01.png'
query = f"""
SELECT AI_COMPLETE('claude-3-5-sonnet',
    '{prompt}',
    TO_FILE('@bears_stage', '{image}'));
"""

session.sql(query)

## Analyse itérative des images

Ici, nous appliquerons Cortex AISQL pour analyser l’image et déterminer les caractéristiques de l’ours.



### Couleur du pelage

Nous allons commencer par analyser la couleur du pelage pour les 200 images.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

prompt = """
Analyze the provided image of a bear. Describe only the fur color of the bear
by choosing the most appropriate term from the following list. The response
should be a single value.
- Light Brown
- Medium Brown
- Blond
- Dark Brown
- Grizzled (A mix of colors with silver-tipped hairs)
- Reddish Brown
- Blackish Brown
- Black
- Brown
- Cinnamon
"""

# Get a list of all image files in the stage
# staged_files_df = session.sql("LIST @bear").collect()

# Sample the first N rows
#nrows = len(df)
nrows=5
staged_files_df = session.sql("LIST @bears_stage").collect()[:nrows]

# Create a list of image filenames to iterate over
image_files = [row['name'] for row in staged_files_df if row['name'].lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]

# Create an empty list to store the results
results_list = []

# Loop through each image file and execute the AI function
for image_path in image_files:
    # Extract just the filename from the full path
    image_name = image_path.split('/')[-1]

    # Dynamically build the query for each image
    query = f"""
    SELECT AI_COMPLETE('claude-3-5-sonnet',
        '{prompt}',
        TO_FILE('@bears_stage', '{image_name}'));
    """

    # Execute the query and collect the result
    result = session.sql(query).collect()
    
    # Append a tuple of the filename and the result to the list
    results_list.append((image_name, result[0][0]))

    print(f"Analysis for {image_name}: {result[0][0]}")

# You can now work with the `results_list`
print("\n--- All Results Collected ---")

In [ ]:
results_list

Supprimons `.PNG` du nom de l’image afin d’obtenir l’identifiant.

In [ ]:
fur_with_id = [
    (image_name.replace('.png', ''), color)
    for image_name, color in results_list
]

fur_with_id

Ici, nous allons convertir les résultats collectés de l’analyse de la couleur du pelage en un DataFrame Snowpark structuré, que nous ajouterons ensuite au dataset complet.


In [ ]:
from snowflake.snowpark import types as T

schema = T.StructType([T.StructField("id", T.StringType()), T.StructField("color", T.StringType())])

# Convert the results_list to a Snowpark DataFrame
df_results = session.create_dataframe(fur_with_id, schema=schema)
df_fur = pd.DataFrame(df_results.to_pandas())
df_fur

### Profil facial

Ensuite, nous analyserons le profil facial des ours, qui est une caractéristique distinctive importante. Le profil facial peut être :
- **Dished** : Profil concave, où l’arête du nez s’incurve
- **Straight** : Profil plat, sans creux entre le front et le nez

In [ ]:
# Define the prompt for facial profile analysis
prompt = """
Analyze the provided image of a bear. Describe only the facial profile of the bear. 
The response must be one of the following two values as a single word with no explanation:
- Dished (Concave profile, where the bridge of the nose dips)
- Straight (Flat profile, with no dip from the forehead to the nose)
"""

# Get a list of first N image files (for testing)
#nrows = len(df)
nrows=5

staged_files_df = session.sql("LIST @bears_stage").collect()[:nrows]

# Create a list of image filenames
image_files = [row['name'] for row in staged_files_df if row['name'].lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]

# Create an empty list to store results
results_list = []

# Process each image
for image_path in image_files:
    image_name = image_path.split('/')[-1]
    
    query = f"""
    SELECT AI_COMPLETE('claude-3-5-sonnet',
        '{prompt}',
        TO_FILE('@bears_stage', '{image_name}'));
    """
    
    result = session.sql(query).collect()
    # Extract ID by removing .png and store with result
    id_value = image_name.replace('.png', '')
    results_list.append((id_value, result[0][0]))
    print(f"Analysis for {image_name}: {result[0][0]}")

# Create Snowpark DataFrame with results
schema = T.StructType([
    T.StructField("ID", T.StringType()), 
    T.StructField("FACIAL_PROFILE", T.StringType())
])
df_results = session.create_dataframe(results_list, schema=schema)

df_facial_profile = pd.DataFrame(df_results.to_pandas())
df_facial_profile

### Texture des coussinets des pattes

Ensuite, nous analyserons la texture des coussinets des pattes des ours, qui est une autre caractéristique distinctive. La texture des coussinets peut être :
- **Smooth** : Moins texturée et relativement plate, pour la marche
- **Rough** : Plus texturée et rainurée, pour l’adhérence et l’escalade

In [ ]:
# Define the prompt for paw pad texture analysis
prompt = """
Analyze the provided image of a bear. Describe only the paw pad texture of the bear. 
The response must be one of the following two values as a single word with no explanation:
- Smooth (Less textured and relatively flat, for walking)
- Rough (More textured and grooved, for gripping and climbing)
"""

# Get a list of first N image files (for testing)
#nrows = len(df)
nrows=5
staged_files_df = session.sql("LIST @bears_stage").collect()[:nrows]

# Create a list of image filenames
image_files = [row['name'] for row in staged_files_df if row['name'].lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]

# Create an empty list to store results
results_list = []

# Process each image
for image_path in image_files:
    image_name = image_path.split('/')[-1]
    
    query = f"""
    SELECT AI_COMPLETE('claude-3-5-sonnet',
        '{prompt}',
        TO_FILE('@bears_stage', '{image_name}'));
    """
    
    result = session.sql(query).collect()
    # Extract ID by removing .png and store with result
    id_value = image_name.replace('.png', '')
    results_list.append((id_value, result[0][0]))
    print(f"Analysis for {image_name}: {result[0][0]}")

# Create Snowpark DataFrame with results
schema = T.StructType([
    T.StructField("ID", T.StringType()), 
    T.StructField("Paw_Pad_Texture", T.StringType())
])
df_results = session.create_dataframe(results_list, schema=schema)

df_paw_pad = pd.DataFrame(df_results.to_pandas())
df_paw_pad

# Opérations sur les données

Dans cette section, nous effectuerons les opérations de données essentielles pour :
- Combiner les caractéristiques extraites (couleur du pelage, profil facial et texture des coussinets) avec le dataset d’origine 
- Écrire le dataset final dans une table Snowflake


In [ ]:
# Read categorical columns
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

# Load the categorical feature data from CSV files
df_fur_color = pd.read_csv("s3://logbrain-datalake/datasets/bears-data/fur_color.csv")
df_facial_profile = pd.read_csv("s3://logbrain-datalake/datasets/bears-data/facial_profile.csv")
df_paw_pad = pd.read_csv("s3://logbrain-datalake/datasets/bears-data/paw_pad_texture.csv")

## Combinaison des caractéristiques

Maintenant que nous avons extrait les trois caractéristiques (couleur du pelage, profil facial et texture des coussinets) à partir des images d’ours, combinons-les avec notre dataset d’origine. 

Le dataset final combiné comprendra :

- Les mesures physiques d’origine (masse corporelle, hauteur de la bosse de l’épaule, etc.)
- L’analyse de la couleur du pelage
- La classification du profil facial
- L’évaluation de la texture des coussinets

Ce dataset enrichi nous donnera une vision plus complète des caractéristiques de chaque ours pour notre analyse.


In [ ]:
# Combining df_fur, df_facial_profile and df_paw_pad to df
# Standardize column names to match
df['id'] = df['id'].str.upper()  # Ensure IDs are in uppercase
df_fur_color['id'] = df_fur_color['id'].str.upper()
df_facial_profile['id'] = df_facial_profile['id'].str.upper()
df_paw_pad['id'] = df_paw_pad['id'].str.upper()

# Perform sequential merges to combine all features using proper indexing
df_combined = df.merge(df_fur_color, on='id', how='inner')
df_combined = df_combined.merge(df_facial_profile, on='id', how='inner')
df_combined = df_combined.merge(df_paw_pad, on='id', how='inner')

# Display the combined DataFrame
df_combined

## Écrire les données dans une table de base de données

### Déterminer la base de données et le schéma actuels

Avant d’écrire dans une table de base de données Snowflake, déterminons l’emplacement actuel de ce notebook, car c’est là que la table de base de données sera créée.

In [ ]:
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA();

In [ ]:
USE DATABASE bears_data;
USE SCHEMA stages;

In [ ]:
df_combined.to_snowflake(
    "BEAR",
    if_exists="replace",
    index=False
)

## Interroger les données depuis la table

In [ ]:
SELECT * FROM BEARS_DATA.STAGES.BEAR;